<a href="https://opendpd.com"><img src="https://raw.githubusercontent.com/lab-emi/OpenDPD/main/pics/OpenDPDlogo_new.png" width="220" alt="OpenDPD"></a>

# OpenDPD tutorial: PA modeling and neural digital pre-distortion, end to end

[![Documentation](https://img.shields.io/badge/docs-lab--emi.github.io%2FOpenDPD-blue)](https://lab-emi.github.io/OpenDPD/)
[![GitHub](https://img.shields.io/badge/GitHub-lab--emi%2FOpenDPD-black)](https://github.com/lab-emi/OpenDPD)
[![PyPI](https://img.shields.io/pypi/v/opendpd)](https://pypi.org/project/opendpd/)
[![Paper](https://img.shields.io/badge/OpenDPD-ISCAS%202024-orange)](https://ieeexplore.ieee.org/abstract/document/10558162)

**OpenDPD** is an open-source PyTorch framework for modeling **power amplifiers (PA)** and learning **digital
pre-distortion (DPD)** on measured data, developed by the
[Lab of Efficient Machine Intelligence](https://www.tudemi.com) at Delft University of Technology. A DPD is a
small network placed *before* the PA; it pre-distorts the signal so that the PA output becomes linear again,
which lowers the spectral regrowth (ACLR) and the modulation error (EVM) of a wideband transmitter.

In this notebook you will

1. look at a measured 200 MHz wideband PA and quantify its distortion (ACLR, EVM, NMSE);
2. train a neural **PA model** and watch it learn (per-epoch plots, GIFs, training curves);
3. train a **DPD** through the frozen PA model and compare *without DPD* vs *with DPD*;
4. export the predistorted signal for a real PA, and fine-tune a **16-bit quantized** DPD;
5. try OpenDPDv2's **TRes-DeltaGRU** with temporal sparsity;
6. build a dataset from **your own CSV**.

**Runtime.** `Runtime → Change runtime type → T4 GPU` trains every model of this notebook for about 20 epochs in
roughly 20 minutes in total; on CPU the notebook lowers the epoch budget automatically. Everything is written to the session's working directory;
the last section shows how to keep what you trained. The full workflow, the datasets, the benchmark and the
API are documented at [https://lab-emi.github.io/OpenDPD/](https://lab-emi.github.io/OpenDPD/).

## 0. Setup

The cell below installs OpenDPD from the `main` branch of the repository, so the notebook always matches the
latest code (the V2.1 plotting system, TRes-DeltaGRU, the benchmark models). The last release on PyPI is
`pip install opendpd`.

In [ ]:
# Latest code from the main branch (the released version is:  !pip install -q opendpd)
!pip install -q "git+https://github.com/lab-emi/OpenDPD.git@main"

In [ ]:
import glob
import json
import os
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython import get_ipython
from IPython.display import Image, display

import opendpd

ACCELERATOR = 'cuda' if torch.cuda.is_available() else 'cpu'
N_EPOCHS = 20 if ACCELERATOR == 'cuda' else 5      # the papers train for 300 epochs; this is a tutorial budget
DATASET = 'DPA_200MHz'                             # a built-in measured dataset (see section 1)
DATASETS_DIR = Path(opendpd.__file__).resolve().parent.parent / 'datasets'   # the datasets shipped with the package

# save/, log/, plots/ and dpd_out/ are written to the working directory
WORKDIR = Path('/content/opendpd_run') if Path('/content').is_dir() else Path.cwd() / 'opendpd_run'
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

gpu = f" ({torch.cuda.get_device_name(0)})" if ACCELERATOR == 'cuda' else ''
print(f"OpenDPD {opendpd.__version__} | torch {torch.__version__} | device: {ACCELERATOR}{gpu}")
print(f"epoch budget: {N_EPOCHS} | working directory: {WORKDIR}")

In [ ]:
# Helpers used throughout the notebook: show saved figures, read the per-epoch logs, plot training curves.

def inline_backend():
    """OpenDPD's plotting module switches matplotlib to the file-only Agg backend; bring the notebook's inline backend back."""
    shell = get_ipython()
    if shell is not None:
        shell.run_line_magic('matplotlib', 'inline')


def show(paths, cols=2, width=6.5):
    """Display PNG figures that OpenDPD wrote under plots/ in a grid."""
    inline_backend()
    paths = [Path(p) for p in paths if Path(p).exists()]
    if not paths:
        print('no figures found')
        return
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(width * cols, width * 0.72 * rows))
    axes = np.ravel(axes)
    for ax, p in zip(axes, paths):
        ax.imshow(plt.imread(p))
        ax.set_title(p.stem, fontsize=10)
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def history(model_path, step):
    """Per-epoch metrics of a run: log/<dataset>/<step>/.../history/<model id>.csv."""
    pattern = f'log/{DATASET}/{step}/**/history/{Path(model_path).stem}.csv'
    return pd.read_csv(glob.glob(pattern, recursive=True)[0])


def plot_history(df, metrics=('TEST_NMSE', 'TEST_EVM', 'TEST_ACLR_AVG')):
    inline_backend()
    fig, axes = plt.subplots(1, len(metrics), figsize=(4.8 * len(metrics), 3.4))
    for ax, m in zip(axes, metrics):
        ax.plot(df['EPOCH'], df[m], marker='o')
        ax.set_xlabel('epoch')
        ax.set_ylabel(m if m.startswith('SP_') else f'{m} (dB)')
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def best_row(df, metric):
    """The epoch OpenDPD kept as the best model: lowest validation value of `metric`."""
    return df.loc[df[metric].idxmin()]

## 1. The data: a measured 200 MHz power amplifier

OpenDPD ships measured datasets. `DPA_200MHz` is a 10-carrier LTE signal (10 × 20 MHz = 200 MHz, 64QAM) sampled
at 800 MS/s through a Doherty PA built in 40 nm CMOS at 2.4 GHz. Each sample is a pair of baseband I/Q values at
the PA input and the time-aligned pair at the PA output; the data is split 60 / 20 / 20 % into train, validation
and test. The other built-in datasets (`DPA_100MHz`, `DPA_160MHz`, `APA_200MHz`, `APA_200MHz_b`) are described in the
[dataset documentation](https://lab-emi.github.io/OpenDPD/datasets/).

In [ ]:
data = opendpd.load_dataset(str(DATASETS_DIR / DATASET))
spec = json.loads((DATASETS_DIR / DATASET / 'spec.json').read_text())
fs, bw, nperseg, n_sub = spec['input_signal_fs'], spec['bw_main_ch'], spec['nperseg'], spec['n_sub_ch']

print(spec['description'])
print(f"{n_sub} x {spec['bw_sub_ch'] / 1e6:.0f} MHz carriers = {bw / 1e6:.0f} MHz, {spec['modulation']}, "
      f"{fs / 1e6:.0f} MS/s, evaluation frames of {nperseg} samples")
for split in ('train', 'val', 'test'):
    print(f"  {split:5s}: {len(data[f'X_{split}']):>6d} samples  (I/Q in: {data[f'X_{split}'].shape}, I/Q out: {data[f'y_{split}'].shape})")

Three views of the same distortion. The **PSD** shows the spectral regrowth in the adjacent channels, the
**AM/AM** curve shows gain compression at high input amplitude, and the **AM/PM** curve shows the phase
rotating with amplitude. The linear reference is the input scaled by the PA's small-signal gain, which is also
the target a DPD tries to reach.

In [ ]:
from scipy.signal import welch
from utils.util import set_target_gain

x, y = data['X_test'], data['y_test']
gain = set_target_gain(data['X_train'], data['y_train'])   # the linear target is gain * input


def psd_db(iq):
    c = iq[:, 0] + 1j * iq[:, 1]
    f, p = welch(c, fs=fs, nperseg=nperseg, return_onesided=False)
    return np.fft.fftshift(f) / 1e6, 10 * np.log10(np.fft.fftshift(p))


amp_in = np.hypot(x[:, 0], x[:, 1])
amp_out = np.hypot(y[:, 0], y[:, 1])
phase = np.degrees(np.angle((y[:, 0] + 1j * y[:, 1]) * np.conj(x[:, 0] + 1j * x[:, 1])))

inline_backend()
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
f_mhz, p_ref = psd_db(gain * x)
axes[0].plot(f_mhz, p_ref - p_ref.max(), label='linear reference (gain · input)', alpha=0.8)
f_mhz, p_out = psd_db(y)
axes[0].plot(f_mhz, p_out - p_ref.max(), label='PA output (measured)', alpha=0.8)
axes[0].set(xlabel='frequency (MHz)', ylabel='PSD (dB)', title='Spectral regrowth')
axes[0].legend()
axes[1].scatter(amp_in, amp_out, s=1, alpha=0.2)
axes[1].plot([0, amp_in.max()], [0, gain * amp_in.max()], 'k--', lw=1, label='linear')
axes[1].set(xlabel='|input|', ylabel='|output|', title='AM/AM')
axes[1].legend()
axes[2].scatter(amp_in, phase, s=1, alpha=0.2)
axes[2].set(xlabel='|input|', ylabel='phase(output / input) (deg)', title='AM/PM')
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

OpenDPD scores every model with the same three metrics, computed on frames of `nperseg` samples of the test
split (`utils.metrics`):

- **NMSE** (dB): normalised mean-square error between the output and the linear target. Lower is better.
- **EVM** (dB): error vector magnitude of the demodulated constellation. Lower is better.
- **ACLR** (dBc): power in the left / right adjacent channel relative to the main channel. Lower is better.

The numbers below are the PA **without any DPD**: keep them in mind, they are the baseline every DPD is
compared against.

In [ ]:
from utils.metrics import ACLR, EVM, NMSE


def metrics_of(pa_output, pa_input):
    """NMSE, EVM and ACLR the way OpenDPD's `plot` step computes them (frames of nperseg samples)."""
    n_seg = len(pa_input) // nperseg
    frames = lambda a: a[:n_seg * nperseg].reshape(n_seg, nperseg, 2)
    out, ref = frames(pa_output), frames(gain * pa_input)
    aclr_l, aclr_r = ACLR(out, fs=fs, nperseg=nperseg, bw_main_ch=bw, n_sub_ch=n_sub)
    return {'NMSE': NMSE(out, ref),
            'EVM': EVM(out, ref, sample_rate=int(fs), bw_main_ch=bw, n_sub_ch=n_sub, nperseg=nperseg),
            'ACLR_L': aclr_l, 'ACLR_R': aclr_r, 'ACLR_AVG': (aclr_l + aclr_r) / 2}


baseline = metrics_of(y, x)
print('PA without DPD (test split):')
for name, value in baseline.items():
    print(f'  {name:9s} {value:8.2f} dB')

## 2. Step 1: learn a behavioral model of the PA

A DPD is trained *through* a model of the PA, so the first step is a neural PA model: a sequence-to-sequence
network that predicts the PA output from the input, trained with backpropagation through time on frames of
`frame_length` samples. The default is a one-layer GRU with 23 hidden units (about 1.9 k parameters); other
backbones are listed by `opendpd-cli --help` (`gru`, `lstm`, `dgru`, `tcn`, `tres_gru`, `gmp`, ...).

`plot=True` switches on the V2.1 plotting system: PSD, AM/AM, AM/PM, constellation, waveform and error plots
for the validation and test split at every `plot_every` epochs, re-rendered with fixed axes at the end, plus
GIF animations, training curves and an HTML dashboard. The best model is selected by validation NMSE.

In [ ]:
pa = opendpd.train_pa(
    dataset_name=DATASET,
    PA_backbone='gru',
    PA_hidden_size=23,
    n_epochs=N_EPOCHS,
    batch_size=64,
    lr=5e-3,
    frame_length=200,
    accelerator=ACCELERATOR,
    seed=0,
    plot=True,
    plot_every=max(1, N_EPOCHS // 10),
    gif_duration=6.0,
)
PA_ID = Path(pa['model_path']).stem
print(f"\nPA model id: {PA_ID}\nweights: {pa['model_path']}\nbest-epoch log: {pa['log_path']}")

Every run has a **model id** that encodes the seed, backbone, hidden size, frame length and parameter count
(`PA_S_0_M_GRU_H_23_F_200_P_1911`). It names the checkpoint under `save/`, the logs under `log/` and the figures
under `plots/`, and it is how the DPD step finds its PA model later. The history log has one row per epoch:

In [ ]:
pa_hist = history(pa['model_path'], 'train_pa')
best = best_row(pa_hist, 'VAL_NMSE')
print(f"best epoch {int(best['EPOCH'])}: test NMSE {best['TEST_NMSE']:.2f} dB, EVM {best['TEST_EVM']:.2f} dB, "
      f"ACLR {best['TEST_ACLR_AVG']:.2f} dBc  (the measured PA output has ACLR {baseline['ACLR_AVG']:.2f} dBc)")
plot_history(pa_hist)

A good PA model reproduces the measured output: the PSD of the prediction lies on top of the measurement
including the regrowth, and the AM/AM and AM/PM curves match. The GIF plays the same figure epoch by epoch.

In [ ]:
PA_PLOTS = Path('plots') / DATASET / 'train_pa' / PA_ID
show([PA_PLOTS / 'best' / name for name in ('psd_test.png', 'amam_test.png', 'ampm_test.png', 'constellation_test.png')])
display(Image(filename=str(PA_PLOTS / 'history' / 'psd_test.gif')))
print(f"all figures: {PA_PLOTS}/  (best/, history/epochs/, history/*.gif, history/dashboard.html, training_curves/)")

## 3. Step 2: train the digital predistorter

The DPD network is placed in front of the frozen PA model. The cascade *DPD → PA model* is trained so that its
output matches the **linear target** (gain × input), again with backpropagation through time; only the DPD
weights are updated. Here the DPD is a GRU with 15 hidden units (about 0.9 k parameters); the PA backbone and
hidden size must be the ones trained above so that the right PA checkpoint is loaded. The best DPD is selected by
validation ACLR.

In [ ]:
dpd = opendpd.train_dpd(
    dataset_name=DATASET,
    DPD_backbone='gru',
    DPD_hidden_size=15,
    PA_backbone='gru',          # the PA model of step 1
    PA_hidden_size=23,
    n_epochs=N_EPOCHS,
    batch_size=64,
    lr=5e-3,
    frame_length=200,
    accelerator=ACCELERATOR,
    seed=0,
    plot=True,
    plot_every=max(1, N_EPOCHS // 10),
    gif_duration=6.0,
)
DPD_ID = Path(dpd['model_path']).stem
print(f"\nDPD model id: {DPD_ID}\nweights: {dpd['model_path']}")

In [ ]:
dpd_hist = history(dpd['model_path'], 'train_dpd')
best = best_row(dpd_hist, 'VAL_ACLR_AVG')
print(f"best epoch {int(best['EPOCH'])}: test ACLR {best['TEST_ACLR_AVG']:.2f} dBc, EVM {best['TEST_EVM']:.2f} dB, "
      f"NMSE {best['TEST_NMSE']:.2f} dB  (without DPD: ACLR {baseline['ACLR_AVG']:.2f} dBc, EVM {baseline['EVM']:.2f} dB)")
plot_history(dpd_hist, metrics=('TEST_ACLR_AVG', 'TEST_EVM', 'TEST_NMSE'))

The overview figure compares the PA output *without* DPD (measured) and *with* DPD (through the PA model) in one
picture; the GIF shows the regrowth being pushed down epoch by epoch. This is the animation on the OpenDPD
website, made from a longer run.

In [ ]:
DPD_PLOTS = Path(glob.glob(f'plots/{DATASET}/train_dpd/*/{DPD_ID}')[0])
show([DPD_PLOTS / 'best' / name for name in ('overview_test.png', 'psd_test.png', 'amam_test.png', 'constellation_test.png')])
display(Image(filename=str(DPD_PLOTS / 'history' / 'overview_test.gif')))

## 4. Step 3: without DPD vs with DPD

`plot_dpd` (the `plot` step of the CLI) loads the PA and DPD checkpoints, runs the test input through the
cascade and prints the three metrics side by side with their improvement, together with comparison figures.
These numbers are computed **through the PA model**, not on hardware: the model is fitted to measurements, so
a good PA model is what makes them meaningful. Reference numbers after 300 epochs, for several model families and
two datasets, are on the [benchmark page](https://lab-emi.github.io/OpenDPD/benchmark/).

In [ ]:
comparison = opendpd.plot_dpd(
    dataset_name=DATASET,
    PA_backbone='gru', PA_hidden_size=23,
    DPD_backbone='gru', DPD_hidden_size=15,
    accelerator=ACCELERATOR,
)
COMPARE = Path('plots') / DATASET / 'compare' / DPD_ID
show([COMPARE / name for name in ('psd_comparison.png', 'metrics_summary.png', 'amam_comparison.png',
                                  'ampm_comparison.png', 'constellation_comparison.png', 'waveform_comparison.png')], cols=3)

In [ ]:
# The same comparison as a table: the measured PA against the best DPD epoch (test split, through the PA model).
best = best_row(dpd_hist, 'VAL_ACLR_AVG')
table = pd.DataFrame({
    'without DPD': [baseline['NMSE'], baseline['EVM'], baseline['ACLR_L'], baseline['ACLR_R']],
    'with DPD': [best['TEST_NMSE'], best['TEST_EVM'], best['TEST_ACLR_L'], best['TEST_ACLR_R']],
}, index=['NMSE (dB)', 'EVM (dB)', 'ACLR left (dBc)', 'ACLR right (dBc)'])
table['improvement (dB)'] = table['without DPD'] - table['with DPD']
display(table.round(2))

## 5. Step 4: the predistorted signal for a real PA

`run_dpd` passes the test input through the trained DPD and writes a CSV with the original I/Q and the
predistorted I/Q. That file is what you play through a signal generator into the physical PA; the captured
output is the real, measured validation of the DPD.

In [ ]:
run = opendpd.run_dpd(
    dataset_name=DATASET,
    DPD_backbone='gru', DPD_hidden_size=15,
    PA_backbone='gru', PA_hidden_size=23,
    accelerator=ACCELERATOR,
    plot=True,
)
dpd_csv = Path('dpd_out') / f'{DPD_ID}.csv'
signal = pd.read_csv(dpd_csv)
print(f"{dpd_csv}: {len(signal)} samples")
display(signal.head())
show([Path('plots') / DATASET / 'run_dpd' / DPD_ID / name for name in ('psd.png', 'amam.png')])

## 6. Quantization-aware training: a 16-bit DPD

Hardware runs fixed-point arithmetic. OpenDPD follows the recipe of the MP-DPD paper: the `qgru` backbone, a
GRU fed with I, Q, |x|² and |x|⁴, is first trained in float and then **fine-tuned with quantization-aware
training**, where the weights and activations are quantized to `n_bits_w` / `n_bits_a` bits in the forward
pass. The quantized run gets its own label (`quant_dir_label`) under `save/`, `log/` and `dpd_out/`, so it
never overwrites the float model. `bash_scripts/OpenDPDv2.sh` applies the same fine-tuning to TRes-DeltaGRU.

In [ ]:
# A: the float qgru DPD (the recipe of step 2 with another backbone)
qgru = opendpd.train_dpd(
    dataset_name=DATASET,
    DPD_backbone='qgru', DPD_hidden_size=15,
    PA_backbone='gru', PA_hidden_size=23,
    n_epochs=max(2, N_EPOCHS // 2),
    batch_size=64, lr=5e-3, frame_length=200,
    accelerator=ACCELERATOR, seed=0,
)

# B: W16A16 quantization-aware fine-tuning from that checkpoint
qat = opendpd.train_dpd(
    dataset_name=DATASET,
    DPD_backbone='qgru', DPD_hidden_size=15,
    PA_backbone='gru', PA_hidden_size=23,
    n_epochs=max(2, N_EPOCHS // 5),             # the quantized cell-based GRU runs step by step and is slower
    batch_size=64, lr=5e-4, frame_length=200,   # fine-tuning: ten times smaller learning rate
    accelerator=ACCELERATOR, seed=0,
    quant=True, n_bits_w=16, n_bits_a=16,
    pretrained_model=qgru['model_path'],
    quant_dir_label='W16A16',
)

rows = {'GRU (step 2)': best_row(dpd_hist, 'VAL_ACLR_AVG'),
        'QGRU float': best_row(history(qgru['model_path'], 'train_dpd'), 'VAL_ACLR_AVG'),
        'QGRU W16A16': best_row(history(qat['model_path'], 'train_dpd'), 'VAL_ACLR_AVG')}
display(pd.DataFrame({name: {'ACLR (dBc)': r['TEST_ACLR_AVG'], 'EVM (dB)': r['TEST_EVM'], 'NMSE (dB)': r['TEST_NMSE']}
                      for name, r in rows.items()}).T.round(2))

In [ ]:
# The predistorted signal of the quantized DPD goes to dpd_out/W16A16/
run_q = opendpd.run_dpd(
    dataset_name=DATASET,
    DPD_backbone='qgru', DPD_hidden_size=15,
    PA_backbone='gru', PA_hidden_size=23,
    accelerator=ACCELERATOR,
    quant=True, n_bits_w=16, n_bits_a=16, quant_dir_label='W16A16',
)
print('quantized predistorted signal:', *glob.glob('dpd_out/W16A16/*.csv'))

## 7. OpenDPDv2: TRes-DeltaGRU and temporal sparsity

OpenDPDv2 introduced **TRes-DeltaGRU**, a temporal-residual GRU that only updates a neuron when its input or
its state changed by more than a threshold (`thx`, `thh`). Small thresholds skip a large share of the
multiply-accumulate operations at almost no cost in linearization, which is what makes an energy-efficient DPD
(the DeltaDPD paper). `collect_delta_stats=True` reports the resulting sparsity. Zero thresholds give the dense
network, the reference model of the OpenDPD benchmark.

In [ ]:
delta = opendpd.train_dpd(
    dataset_name=DATASET,
    DPD_backbone='tres_deltagru', DPD_hidden_size=15,
    PA_backbone='gru', PA_hidden_size=23,
    n_epochs=max(2, N_EPOCHS // 2),
    batch_size=64,
    lr=5e-3,
    accelerator=ACCELERATOR,
    seed=0,
    thx=0.02, thh=0.02,                   # delta thresholds on the input and the hidden state
    collect_delta_stats=True,
)
d_hist = history(delta['model_path'], 'train_dpd')
d_best = best_row(d_hist, 'VAL_ACLR_AVG')
print(f"\nTRes-DeltaGRU (thx = thh = 0.02): test ACLR {d_best['TEST_ACLR_AVG']:.2f} dBc, EVM {d_best['TEST_EVM']:.2f} dB")
print(f"temporal sparsity at the best epoch: {d_best['SP_T_DX']:.0%} of the input deltas and "
      f"{d_best['SP_T_DH']:.0%} of the hidden-state deltas are below the threshold and skipped "
      f"({d_best['SP_T_DV']:.0%} of all multiply-accumulates)")
plot_history(d_hist, metrics=('TEST_ACLR_AVG', 'SP_T_DX', 'SP_T_DH'))

## 8. Your own PA: a dataset from a CSV

Measured data arrives as one CSV with the columns `I_in, Q_in, I_out, Q_out` (time-aligned input and output of
the PA). `create_dataset` splits it and writes a dataset folder with its `spec.json` next to the built-in ones,
after which the new name works everywhere. The spec must state the sampling rate, the channel bandwidths and the
frame length used for the metrics; the
[dataset documentation](https://lab-emi.github.io/OpenDPD/datasets/) explains every field. The example CSV of the
repository stands in for your measurement here.

In [ ]:
csv_path = Path('my_pa_capture.csv')
urllib.request.urlretrieve('https://raw.githubusercontent.com/lab-emi/OpenDPD/main/examples/single_csv_format_example.csv', csv_path)
print(pd.read_csv(csv_path, nrows=3))

my_dataset = opendpd.create_dataset(
    csv_path=str(csv_path),
    output_dir=str(DATASETS_DIR),       # next to the built-in datasets, so dataset_name='MyCustomPA' resolves
    dataset_name='MyCustomPA',
    train_ratio=0.6, val_ratio=0.2, test_ratio=0.2,
    input_signal_fs=800e6, bw_main_ch=200e6, bw_sub_ch=20e6, n_sub_ch=10, nperseg=2560,
)
print(json.loads((Path(my_dataset) / 'spec.json').read_text()))

In [ ]:
mine = opendpd.train_pa(
    dataset_name='MyCustomPA',
    PA_backbone='gru', PA_hidden_size=23,
    n_epochs=max(2, N_EPOCHS // 4),
    batch_size=64, lr=5e-3,
    accelerator=ACCELERATOR, seed=0,
)
mine_hist = pd.read_csv(glob.glob(f"log/MyCustomPA/train_pa/**/history/{Path(mine['model_path']).stem}.csv", recursive=True)[0])
print(f"PA model on MyCustomPA: test NMSE {best_row(mine_hist, 'VAL_NMSE')['TEST_NMSE']:.2f} dB after {len(mine_hist)} epochs")

## 9. Keep what you trained

A Colab session is temporary. Download the checkpoints, logs and figures, or copy them to Google Drive:

```python
import shutil
shutil.make_archive('/content/opendpd_results', 'zip', WORKDIR)   # save/, log/, plots/, dpd_out/
from google.colab import files
files.download('/content/opendpd_results.zip')
```

```python
from google.colab import drive
drive.mount('/content/drive')
shutil.copytree(WORKDIR, '/content/drive/MyDrive/opendpd_run', dirs_exist_ok=True)
```

## 10. Where to go next

Every function you called has a command-line twin, which is what the scripts of the repository use:

| Python API | Command line (`opendpd-cli` or `python main.py`) |
|---|---|
| `opendpd.train_pa(dataset_name=..., PA_backbone='gru', plot=True)` | `opendpd-cli --step train_pa --dataset_name DPA_200MHz --PA_backbone gru --plot` |
| `opendpd.train_dpd(dataset_name=..., DPD_backbone='gru')` | `opendpd-cli --step train_dpd --dataset_name DPA_200MHz --DPD_backbone gru` |
| `opendpd.plot_dpd(dataset_name=...)` | `opendpd-cli --step plot --dataset_name DPA_200MHz` |
| `opendpd.run_dpd(dataset_name=...)` | `opendpd-cli --step run_dpd --dataset_name DPA_200MHz` |
| `..., quant=True, n_bits_w=16, n_bits_a=16, pretrained_model=...` | `... --quant --n_bits_w 16 --n_bits_a 16 --pretrained_model ...` |

- **Documentation**: [installation](https://lab-emi.github.io/OpenDPD/install/), [end-to-end training](https://lab-emi.github.io/OpenDPD/training/),
  [datasets](https://lab-emi.github.io/OpenDPD/datasets/), [benchmark](https://lab-emi.github.io/OpenDPD/benchmark/), [API reference](https://lab-emi.github.io/OpenDPD/api/).
- **Reproduce the papers**: `bash_scripts/` in the repository trains every model of OpenDPDv1, MP-DPD and
  OpenDPDv2 with the published recipes ([guide](https://lab-emi.github.io/OpenDPD/reproducing/)).
- **Contribute** a backbone, a pre-trained model or a measured dataset: [https://github.com/lab-emi/OpenDPD](https://github.com/lab-emi/OpenDPD).
- **Cite** OpenDPD ([all references](https://lab-emi.github.io/OpenDPD/about/)):

```bibtex
@INPROCEEDINGS{Wu2024ISCAS,
  author={Wu, Yizhuo and Singh, Gagan Deep and Beikmirza, Mohammadreza and de Vreede, Leo C. N. and Alavi, Morteza and Gao, Chang},
  booktitle={2024 IEEE International Symposium on Circuits and Systems (ISCAS)},
  title={OpenDPD: An Open-Source End-to-End Learning & Benchmarking Framework for Wideband Power Amplifier Modeling and Digital Pre-Distortion},
  year={2024}, pages={1-5}, doi={10.1109/ISCAS58744.2024.10558162}}
```